In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. 定义超参数与特殊标记 ---
SOS_token = 0  # 句子开始标记 (Start of Sentence)
EOS_token = 1  # 句子结束标记 (End of Sentence)
VOCAB_SIZE = 1000 # 假设词表大小为1000
HIDDEN_SIZE = 256
MAX_LENGTH = 10   # 最大生成长度

In [2]:
# --- 2. 定义极简版 Decoder ---
class SimpleDecoder(nn.Module):
    def __init__(self, hidden_size, vocab_size):
        super(SimpleDecoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_step, hidden):
        # input_step shape: (1, 1) -> [batch_size, seq_len]
        embedded = self.embedding(input_step)
        output, hidden = self.gru(embedded, hidden)
        # output shape: (1, 1, hidden_size)
        prediction = self.out(output.squeeze(1)) # shape: (1, vocab_size)
        return prediction, hidden

In [3]:
# --- 2. 定义极简版 Decoder ---
class SimpleDecoder(nn.Module):
    def __init__(self, hidden_size, vocab_size):
        super(SimpleDecoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_step, hidden):
        # input_step shape: (1, 1) -> [batch_size, seq_len]
        embedded = self.embedding(input_step)
        output, hidden = self.gru(embedded, hidden)
        # output shape: (1, 1, hidden_size)
        prediction = self.out(output.squeeze(1)) # shape: (1, vocab_size)
        return prediction, hidden

# --- 3. 束搜索核心逻辑 ---
def beam_search_decode(decoder, initial_hidden, beam_width=3, max_length=MAX_LENGTH):
    """
    参数:
        decoder: 解码器模型
        initial_hidden: 编码器最后输出的隐藏状态 (作为解码器的初始状态)
        beam_width: 束宽 k
        max_length: 句子的最大生成长度
    """
    
    # beam 列表中的元素格式为: (累积对数概率分数, 单词索引列表, 当前隐藏状态)
    # 初始化：得分为 0.0，序列只有 <SOS>，隐藏状态来自 Encoder
    beams = [(0.0, [SOS_token], initial_hidden)]
    
    # 存放已经生成 <EOS> 的完整句子
    completed_beams = []

    for step in range(max_length):
        new_beams = []
        
        # 遍历当前保留的所有候选分支
        for score, seq, hidden in beams:
            # 如果该分支已经结束，直接放入完成列表，不再向后扩展
            if seq[-1] == EOS_token:
                completed_beams.append((score, seq))
                continue

            # 准备当前步的输入单词 (取序列的最后一个词)
            dec_input = torch.tensor([[seq[-1]]], dtype=torch.long)
            
            # 前向传播，获取预测结果和新的隐藏状态
            out, next_hidden = decoder(dec_input, hidden)
            
            # 将输出转化为对数概率 (Log Softmax) 以防止下溢，并且方便相加
            log_probs = F.log_softmax(out, dim=1)
            
            # 选出当前步概率最大的前 k 个词汇
            topk_log_probs, topk_indices = torch.topk(log_probs, beam_width)
            
            # 将这 k 个选择分别与当前分支合并，形成 k 个新的分支
            for i in range(beam_width):
                new_score = score + topk_log_probs[0][i].item() # 累加分数
                new_seq = seq + [topk_indices[0][i].item()]     # 追加新词
                new_beams.append((new_score, new_seq, next_hidden))
        
        # 如果所有保留的分支都已经遇到了 EOS，则提前停止
        if len(new_beams) == 0:
            break

        # 将所有新生成的分支按照分数从高到低排序
        new_beams = sorted(new_beams, key=lambda x: x[0], reverse=True)
        
        # 核心：只保留排名前 k (beam_width) 的分支，丢弃其余的 (剪枝)
        beams = new_beams[:beam_width]

    # 循环结束后，把还没有遇到 <EOS> 但达到了最大长度的句子也加入完成列表
    for score, seq, _ in beams:
        if seq[-1] != EOS_token:
            completed_beams.append((score, seq))

    # --- 4. 长度惩罚 (可选但推荐) ---
    # 为了防止模型偏好过短的句子，我们用序列长度对分数进行惩罚/归一化
    # Score = total_log_prob / (length ^ alpha)
    alpha = 0.7
    normalized_beams = []
    for score, seq in completed_beams:
        # 不计算 <SOS> 标记的长度
        length = len(seq) - 1 if len(seq) > 1 else 1 
        norm_score = score / (length ** alpha)
        normalized_beams.append((norm_score, seq))

    # 按照归一化后的分数再次排序，选出最优解
    normalized_beams = sorted(normalized_beams, key=lambda x: x[0], reverse=True)
    best_score, best_seq = normalized_beams[0]
    
    return best_seq, best_score

In [4]:
# --- 5. 模拟运行 ---
# 实例化 Decoder
decoder = SimpleDecoder(HIDDEN_SIZE, VOCAB_SIZE)
decoder.eval() # 切换到推理模式

# 模拟 Encoder 输出的初始隐藏状态 shape: (num_layers, batch_size, hidden_size)
encoder_hidden = torch.randn(1, 1, HIDDEN_SIZE)

    # 执行束搜索
best_sequence, sequence_score = beam_search_decode(
    decoder, 
    encoder_hidden, 
    beam_width=3, 
    max_length=15
)

print(f"最优生成序列 (索引): {best_sequence}")
print(f"序列归一化得分: {sequence_score:.4f}")

最优生成序列 (索引): [0, 668, 126, 126, 569, 260, 690, 547, 547, 475, 260, 309, 105, 467, 547, 475]
序列归一化得分: -14.1308
